***SCAN training***

In [ ]:
#training setup
model = Transformer(
    src_vocab_size=len(src_vocab),
    tgt_vocab_size=len(tgt_vocab),
    src_pad_idx=PAD,
    tgt_pad_idx=PAD,
    emb_dim=128,
    num_layers=2,
    num_heads=4,
    forward_dim=256,
    dropout=0.1,
    max_len=100,
).to(device)

criterion = torch.nn.CrossEntropyLoss(ignore_index=PAD)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

#accuracy
def tokens_accuracy(logits, targets):
    """
    logits: [B, L, V]
    targets: [B, L]
    """
    preds = logits.argmax(dim=-1)
    mask = (targets != PAD)
    correct = (preds == targets) & mask
    return correct.sum().item() / mask.sum().item()

#validation loop
@torch.no_grad()
def evaluate():
    model.eval()
    total_loss = 0
    total_acc = 0
    steps = 0

    for src, tgt_in, tgt_out in test_dl:
        src, tgt_in, tgt_out = src.to(device), tgt_in.to(device), tgt_out.to(device)

        logits = model(src, tgt_in)  # [B, L, V]

        loss = criterion(
            logits.reshape(-1, logits.size(-1)),
            tgt_out.reshape(-1)
        )

        acc = tokens_accuracy(logits, tgt_out)

        total_loss += loss.item()
        total_acc += acc
        steps += 1

    return total_loss / steps, total_acc / steps


#training loop

EPOCHS = 20

for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss = 0
    steps = 0

    for src, tgt_in, tgt_out in train_dl:
        src, tgt_in, tgt_out = src.to(device), tgt_in.to(device), tgt_out.to(device)

        optimizer.zero_grad()

        logits = model(src, tgt_in)   # [B, L, V]

        loss = criterion(
            logits.reshape(-1, logits.size(-1)),
            tgt_out.reshape(-1)
        )

        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        steps += 1

    val_loss, val_acc = evaluate()

    print(f"Epoch {epoch:02d} | "
          f"train_loss={running_loss/steps:.4f} | "
          f"val_loss={val_loss:.4f} | "
          f"val_acc={val_acc*100:.2f}%")


Epoch 01 | train_loss=0.9811 | val_loss=0.6459 | val_acc=74.20%
Epoch 02 | train_loss=0.5691 | val_loss=0.4346 | val_acc=81.47%
Epoch 03 | train_loss=0.3619 | val_loss=0.2343 | val_acc=90.48%
Epoch 04 | train_loss=0.2316 | val_loss=0.1461 | val_acc=94.02%
Epoch 05 | train_loss=0.1609 | val_loss=0.1037 | val_acc=95.55%
Epoch 06 | train_loss=0.1159 | val_loss=0.0626 | val_acc=97.50%
Epoch 07 | train_loss=0.0860 | val_loss=0.0520 | val_acc=97.85%
Epoch 08 | train_loss=0.0686 | val_loss=0.0390 | val_acc=98.46%
Epoch 09 | train_loss=0.0552 | val_loss=0.0257 | val_acc=99.12%
Epoch 10 | train_loss=0.0475 | val_loss=0.0397 | val_acc=98.25%
Epoch 11 | train_loss=0.0393 | val_loss=0.0277 | val_acc=98.95%
Epoch 12 | train_loss=0.0332 | val_loss=0.0249 | val_acc=98.96%
Epoch 13 | train_loss=0.0303 | val_loss=0.0141 | val_acc=99.49%
Epoch 14 | train_loss=0.0264 | val_loss=0.0155 | val_acc=99.45%
Epoch 15 | train_loss=0.0232 | val_loss=0.0306 | val_acc=98.78%
Epoch 16 | train_loss=0.0203 | val_loss=

In [ ]:
torch.save(model.state_dict(), "transformer_scan.pt")